# Business Entity Resolution: Labeled Pair Generator
## Pre-Filter Layer (Blocking) & Training Pair Generation for Two-Tower Model

---

### 1. Problem Overview
In business entity resolution, we want to determine whether records from different data sources refer to the exact same real-world company.
- **Source 1 entities ($N$):** ~2.2 Million
- **Source 2 entities ($A$):** ~5.0 Million
- **Source 3 entities ($B$):** ~5.3 Million

#### The Scale Problem: Why Naive Comparison Fails
If we compared every entity in Source 1 against all entities in Source 2 and Source 3:
$$\text{Total Pairs} = N \times (A + B) = 2.2\text{M} \times 10.3\text{M} \approx 22.8\text{ Trillion Pairs}$$
Saving 22.8 trillion pairs would take **over 900 Terabytes** of disk space and months of compute time!

#### The Solution: Pre-Filter Layer (Blocking)
Instead of comparing everything, we use **rule-based blocking filters**:
1. **Layer 0 (Country Partition):** An entity in the US never matches an entity in India. Verified on ground truth: **100% of matches share the exact same country**.
2. **Layer 1 (Token Inverted Index):** We index entities by their normalized name tokens (stripping legal suffixes like *Inc, LLC, Ltd, Pvt* and common stopwords). We query candidates using the **rarest token** (most unique word).
3. **Layer 1b (Character Trigram Fallback):** For entities with typos, merged words, or website domains (e.g. `landcruzhilliard.com`), we match using overlapping 3-character substrings.

#### Generating Training Labels:
- **Positive Pairs (`label = 1`):** True matches directly from `train_ground_truth.tsv` (~7.6M pairs).
- **Hard Negative Pairs (`label = 0`):** Candidates retrieved by the Pre-Filter that are in the same country and share words, but are **NOT** true matches in the ground truth.


### Setup and Imports
We import standard data science packages and our custom Pre-Filter module (`blocking_index.py`).


In [9]:
import os
import sys
import csv
import time
import random
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from typing import List, Set, Dict, Tuple, Optional, Any

# Ensure project root is in Python sys.path so we can import our modules
# If this notebook is running inside 'src/data_preprocessing', project root is two levels up: ../../
current_dir = os.getcwd()
if os.path.exists(os.path.join(current_dir, "blocking_index.py")):
    # Current directory is src/data_preprocessing
    src_dir = current_dir
    project_root = os.path.abspath(os.path.join(current_dir, "../../"))
else:
    # Running from project root
    project_root = current_dir
    src_dir = os.path.join(project_root, "src", "data_preprocessing")

if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Import our Pre-Filter Layer and helper functions
from blocking_index import (
    BlockingIndex,
    BlockingConfig,
    normalize_tokens,
    extract_char_trigrams,
    load_ground_truth,
    sample_hard_negatives,
)

print(f"[✓] Project root : {project_root}")
print(f"[✓] Source dir   : {src_dir}")
print(f"[✓] Successfully loaded BlockingIndex module!")


[✓] Project root : c:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution
[✓] Source dir   : c:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\src\data_preprocessing
[✓] Successfully loaded BlockingIndex module!


### File Paths & Hyperparameters
Here we define the input dataset locations, output file destination, and sampling parameters.


In [10]:
# Dataset input paths
DATASET_DIR = os.path.join(project_root, "dataset", "train")
GROUND_TRUTH_PATH = os.path.join(DATASET_DIR, "train_ground_truth.tsv")
SOURCE1_PATH = os.path.join(DATASET_DIR, "train_source1.tsv")
SOURCE2_PATH = os.path.join(DATASET_DIR, "train_source2.tsv")
SOURCE3_PATH = os.path.join(DATASET_DIR, "train_source3.tsv")

# Output path for the generated labeled pairs
OUTPUT_DIR = os.path.join(project_root, "data", "pre-processing")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_TSV_PATH = os.path.join(OUTPUT_DIR, "labeledPairs.tsv")

# Hyperparameters:
# NEG_RATIO: Number of hard negative pairs (label=0) to sample for each positive pair (label=1)
# A 5:1 ratio gives a balanced dataset that helps the Two-Tower model learn fine distinctions.
NEG_RATIO = 5

# RANDOM_SEED: For reproducible negative sampling
RANDOM_SEED = 42

# CHUNK_SIZE: How many rows to write to disk at a time during streaming.
# Writing in chunks prevents RAM usage from exploding!
CHUNK_SIZE = 50_000

print(f"Input Ground Truth : {GROUND_TRUTH_PATH}")
print(f"Output Pairs TSV   : {OUTPUT_TSV_PATH}")
print(f"Negative Ratio     : {NEG_RATIO} negatives per positive")


Input Ground Truth : c:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\dataset\train\train_ground_truth.tsv
Output Pairs TSV   : c:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv
Negative Ratio     : 5 negatives per positive


## Step 1: Load Ground Truth Matches
`train_ground_truth.tsv` contains the true entity mappings.
Each line has:
- `source1_entity_id`: The ID of the Source 1 company (e.g. `S1-965667`).
- `matched_entity_ids`: Comma-separated IDs of matching entities in Source 2 and Source 3 (e.g. `S2-681193310,S3-775321672`).
- If `matched_entity_ids` is empty or NaN, the entity is a **singleton** (it has no match in S2 or S3).

All positive pairs will receive `label = 1`.


In [11]:
print("[*] Loading Ground Truth matches from TSV...")
t0 = time.time()

# load_ground_truth streams the file safely without crashing on quotes
gt_map, gt_stats = load_ground_truth(GROUND_TRUTH_PATH)
t_load = time.time() - t0

print(f"[✓] Ground Truth loaded in {t_load:.2f} seconds!")
print(f"    - Total Source 1 entities : {gt_stats['total_s1_entities']:,}")
print(f"    - Entities with matches   : {gt_stats['entities_with_matches']:,}")
print(f"    - Singletons (no matches) : {gt_stats['singletons']:,}")
print(f"    - Total Positive Pairs    : {gt_stats['total_positive_pairs']:,}")

# Peek at a sample mapping
sample_s1 = next(iter(gt_map))
print(f"\nSample S1 mapping: {sample_s1} -> {list(gt_map[sample_s1])[:3]}...")


[*] Loading Ground Truth matches from TSV...
[✓] Ground Truth loaded in 9.95 seconds!
    - Total Source 1 entities : 2,206,821
    - Entities with matches   : 2,083,574
    - Singletons (no matches) : 123,247
    - Total Positive Pairs    : 7,638,364

Sample S1 mapping: S1-965667 -> ['S3-775321672', 'S2-743505751', 'S3-860443364']...


## Step 2: Build the Pre-Filter Blocking Index
Next, we index all entities from **Source 2** (`train_source2.tsv`) and **Source 3** (`train_source3.tsv`).

### How the Blocking Index operates:
1. **Country Partition (Layer 0):** Groups entities into country buckets (e.g. `US`, `India`).
2. **Name Tokenization & Cleaning (Layer 1):** Strips punctuation, lowercases text, and filters out common stopwords and legal suffixes (`Inc`, `LLC`, `Ltd`, `Pvt Ltd`, etc.).
3. **Rarest Token Querying:** For any query name, the index identifies the **rarest token** (the word that appears least often across the dataset, like `"Prabhav"` or `"Orelee"`). This shrinks candidate lists from millions to only a few hundred!

> **Memory Tip:** `enable_trigram_fallback=False` uses ~1.5 GB RAM while maintaining ~85.5% recall. Setting it to `True` adds character trigrams (~91.9% recall) but requires more RAM.


In [12]:
# Configure the Pre-Filter Blocking Index
config = BlockingConfig(
    min_token_length=2,             # Ignore 1-letter tokens
    top_k_rarest_tokens=1,          # Retrieve candidates sharing the single rarest token
    enable_trigram_fallback=False,  # Memory-safe token blocking (set True if you have >=8 GB RAM)
    neg_ratio=NEG_RATIO,
    random_seed=RANDOM_SEED
)

blocking_index = BlockingIndex(config=config)

print("[*] Building Pre-Filter Index from Source 2 and Source 3...")
print("    (Reading TSVs line-by-line using streaming reader)...")
t0 = time.time()

# Tip: To run a quick test on a smaller sample, you can set max_entities=100_000
# For full training data generation, leave max_entities=None
blocking_index.build_from_tsv(
    SOURCE2_PATH,
    SOURCE3_PATH,
    show_progress=True
)

t_build = time.time() - t0
print(f"[✓] Pre-Filter Index built in {t_build:.2f} seconds!")

# View index summary statistics
index_stats = blocking_index.stats()
print(f"    - Total entities indexed   : {index_stats['total_entities']:,}")
print(f"    - Entities by country      : {index_stats['countries']}")
print(f"    - Total unique word tokens : {index_stats['total_unique_tokens']:,}")


[*] Building Pre-Filter Index from Source 2 and Source 3...
    (Reading TSVs line-by-line using streaming reader)...
[*] Indexing entities from: train_source2.tsv ...
    Indexed 500,000 entities from train_source2.tsv (Total: 500,000)
    Indexed 1,000,000 entities from train_source2.tsv (Total: 1,000,000)
    Indexed 1,500,000 entities from train_source2.tsv (Total: 1,500,000)
    Indexed 2,000,000 entities from train_source2.tsv (Total: 2,000,000)
    Indexed 2,500,000 entities from train_source2.tsv (Total: 2,500,000)
    Indexed 3,000,000 entities from train_source2.tsv (Total: 3,000,000)
    Indexed 3,500,000 entities from train_source2.tsv (Total: 3,500,000)
    Indexed 4,000,000 entities from train_source2.tsv (Total: 4,000,000)
    Indexed 4,500,000 entities from train_source2.tsv (Total: 4,500,000)
    Indexed 5,000,000 entities from train_source2.tsv (Total: 5,000,000)
[+] Finished train_source2.tsv: 5,034,616 entities processed.
[*] Indexing entities from: train_source3.ts

## Step 3: Interactive Demo & Visual Inspection
Let's verify how the Pre-Filter retrieves candidates and samples hard negatives for **5 sample Source 1 entities**.
We will look at:
- S1 Entity details (Name, Address, Country)
- True positive matches (`label = 1`)
- Retrieved candidates from the Blocking Index
- Sampled hard negatives (`label = 0`)

Notice that hard negatives are from the **same country** and share **similar name tokens**, but are distinct companies. This teaches the model to look at address, subtle spelling, and street details!


In [13]:
# Fast helper to fetch entity details from TSVs by ID
def fetch_details_by_id(tsv_path: str, target_ids: Set[str]) -> Dict[str, Dict[str, str]]:
    details = {}
    if not target_ids or not os.path.exists(tsv_path):
        return details
    with open(tsv_path, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
        header = next(reader, None)
        for row in reader:
            if row and row[0] in target_ids:
                details[row[0]] = {
                    "id": row[0],
                    "name": row[1] if len(row) > 1 else "",
                    "address": row[2] if len(row) > 2 else "",
                    "country": row[3] if len(row) > 3 else ""
                }
                if len(details) == len(target_ids):
                    break
    return details

# Pick 5 sample entities: 3 with matches, 2 singletons
sample_s1_ids = []
for s1_id, matches in gt_map.items():
    if len(matches) >= 2 and len(sample_s1_ids) < 3:
        sample_s1_ids.append(s1_id)
    elif len(matches) == 0 and len(sample_s1_ids) >= 3 and len(sample_s1_ids) < 5:
        sample_s1_ids.append(s1_id)
    if len(sample_s1_ids) == 5:
        break

# Fetch records for these 5 sample S1 entities
s1_sample_records = fetch_details_by_id(SOURCE1_PATH, set(sample_s1_ids))

print("=" * 85)
print("INTERACTIVE DEMO: CANDIDATE RETRIEVAL & HARD NEGATIVE SAMPLING")
print("=" * 85)

rng = random.Random(RANDOM_SEED)

for i, s1_id in enumerate(sample_s1_ids, 1):
    s1_data = s1_sample_records.get(s1_id, {"name": "N/A", "address": "N/A", "country": "N/A"})
    positives = gt_map.get(s1_id, set())

    # Retrieve candidates using the Blocking Index
    candidates = blocking_index.get_candidates(name=s1_data["name"], country=s1_data["country"])

    # Sample hard negatives
    needed_negs = max(2, len(positives) * NEG_RATIO) if positives else 2
    sampled_negs = sample_hard_negatives(candidates, positives, num_to_sample=needed_negs, rng=rng)

    # Fetch details for positive matches and sampled negatives
    all_needed_ids = positives | set(sampled_negs)
    target_info = {}
    target_info.update(fetch_details_by_id(SOURCE2_PATH, all_needed_ids))
    target_info.update(fetch_details_by_id(SOURCE3_PATH, all_needed_ids))

    print(f"\n[{i}] Query S1: {s1_id}")
    print(f"    Name    : {s1_data['name']}")
    print(f"    Country : {s1_data['country']} | Address: {s1_data['address']}")
    print(f"    Blocking Candidates Retrieved: {len(candidates)}")

    print("    --> Positive Matches (label = 1):")
    if positives:
        for pos_id in sorted(positives):
            rec = target_info.get(pos_id, {"name": "N/A", "address": "N/A", "country": ""})
            print(f"        [+] {pos_id} | {rec['name']} | {rec['address']} ({rec['country']})")
    else:
        print("        (None - Singleton Entity)")

    print("    --> Sampled Hard Negatives (label = 0, showing up to 3):")
    if sampled_negs:
        for neg_id in sampled_negs[:3]:
            rec = target_info.get(neg_id, {"name": "N/A", "address": "N/A", "country": ""})
            print(f"        [-] {neg_id} | {rec['name']} | {rec['address']} ({rec['country']})")
    else:
        print("        (No negative candidates found)")
    print("-" * 85)


INTERACTIVE DEMO: CANDIDATE RETRIEVAL & HARD NEGATIVE SAMPLING

[1] Query S1: S1-965667
    Name    : Maure Williams Colombier Inc
    Country : US | Address: 85 Wayne Avenue, Ticonderoga, NY
    Blocking Candidates Retrieved: 123
    --> Positive Matches (label = 1):
        [+] S2-681193310 | Maure Wilblims Colombier Inc |  (US)
        [+] S2-743505751 | Maure Williams Colombier |  (US)
        [+] S3-11291185 | maurewilliamscolombier.com | Wayne Ave, Ticonderoga Townshiip, New York (US)
        [+] S3-775321672 | Dréxkor | 85 Wanye Avenue, Ticonderoga Townshiip, New York (US)
        [+] S3-860443364 | Maure Williams Inc Center |  (US)
    --> Sampled Hard Negatives (label = 0, showing up to 3):
        [-] S3-836112694 | Arcdrexflux a/k/a Maure Long Summit Flexible Co | 823 3rd Street, Columbus, MT (US)
        [-] S2-478851392 | Maure Snider, CPA LP | 349 MAIN SDREET, VASSALBOROCITY, ME (US)
        [-] S2-134304118 | Maure's United Náils | 19752-19754 SUNSPOT WAY, MARICOPA, AZ (

## Step 4: Stream Labeled Pairs to TSV
Now we generate the complete `labeledPairs.tsv` file.

### Why Streaming?
With 2.2 million Source 1 entities and ~7.6 million positive pairs, sampling 5 negatives per positive produces **~46 million labeled pairs**.
- If we stored 46 million pairs in memory, Python would consume **~4 to 8 GB RAM** and crash.
- Instead, we **stream-write** rows in batches of `CHUNK_SIZE = 50,000` directly to disk.
- Memory usage stays tiny (< 200 MB), and performance is very high!

### Output Schema:
| Column | Description |
|---|---|
| `source1_entity_id` | ID of the Source 1 entity (e.g. `S1-965667`) |
| `target_entity_id` | ID of the matching or candidate entity from Source 2 or 3 |
| `label` | `1` for true positive match, `0` for hard negative candidate |


In [14]:
def generate_labeled_pairs_tsv(
    s1_tsv_path: str,
    ground_truth_map: Dict[str, Set[str]],
    index: BlockingIndex,
    output_tsv_path: str,
    neg_ratio: int = 5,
    singleton_neg_count: int = 2,
    chunk_size: int = 50_000,
    max_s1_entities: Optional[int] = None,
    random_seed: int = 42
) -> Dict[str, Any]:
    """
    Generates labeledPairs.tsv by streaming rows directly to disk.
    
    For each Source 1 entity:
    1. Writes all true positive pairs (label = 1) from ground truth.
    2. Queries the BlockingIndex to retrieve candidate entity IDs in the same country.
    3. Samples hard negatives (candidates not in ground truth) and writes them (label = 0).
    4. Flushes rows to disk every chunk_size rows to keep RAM usage minimal.
    """
    rng = random.Random(random_seed)
    os.makedirs(os.path.dirname(os.path.abspath(output_tsv_path)), exist_ok=True)

    positives_count = 0
    negatives_count = 0
    entities_count = 0
    start_time = time.time()

    with open(s1_tsv_path, "r", encoding="utf-8", errors="ignore") as f_in, \
         open(output_tsv_path, "w", encoding="utf-8", newline="") as f_out:

        reader = csv.reader(f_in, delimiter="\t", quoting=csv.QUOTE_NONE)
        writer = csv.writer(f_out, delimiter="\t", lineterminator="\n")

        # Write TSV header
        writer.writerow(["source1_entity_id", "target_entity_id", "label"])
        header = next(reader, None)

        buffer: List[List[Any]] = []

        for row in reader:
            if not row:
                continue

            s1_id = row[0].strip()
            s1_name = row[1].strip() if len(row) > 1 else ""
            s1_country = row[3].strip() if len(row) > 3 else ""

            # 1. Add true positive pairs (label = 1) from Ground Truth
            positives = ground_truth_map.get(s1_id, set())
            for pos_id in positives:
                buffer.append([s1_id, pos_id, 1])
                positives_count += 1

            # 2. Retrieve candidates from the Blocking Index
            candidates = index.get_candidates(name=s1_name, country=s1_country)

            # 3. Sample hard negatives (label = 0)
            if positives:
                num_negs = len(positives) * neg_ratio
            else:
                num_negs = singleton_neg_count

            sampled_negs = sample_hard_negatives(candidates, positives, num_to_sample=num_negs, rng=rng)
            for neg_id in sampled_negs:
                buffer.append([s1_id, neg_id, 0])
                negatives_count += 1

            entities_count += 1

            # Flush buffer to disk when chunk_size is reached
            if len(buffer) >= chunk_size:
                writer.writerows(buffer)
                buffer.clear()
                elapsed = time.time() - start_time
                rate = entities_count / max(elapsed, 0.001)
                print(f"    Entities: {entities_count:,} | Positives: {positives_count:,} | Negatives: {negatives_count:,} ({rate:.0f} ent/sec)")

            if max_s1_entities and entities_count >= max_s1_entities:
                print(f"[!] Reached max_s1_entities limit: {max_s1_entities:,}")
                break

        # Write any remaining rows in the buffer
        if buffer:
            writer.writerows(buffer)
            buffer.clear()

    total_time = time.time() - start_time
    total_pairs = positives_count + negatives_count

    return {
        "entities_processed": entities_count,
        "total_positives": positives_count,
        "total_negatives": negatives_count,
        "total_pairs": total_pairs,
        "duration_seconds": total_time,
        "output_file": output_tsv_path
    }


### Execute Labeled Pair Generation
Run the cell below to generate `data/pre-processing/labeledPairs.tsv`.

> **Control the run size:**
> - To test quickly on a small subset (e.g. 50,000 entities in ~15 seconds), set `MAX_ENTITIES = 50_000`.
> - To run on all 2.2 million entities, set `MAX_ENTITIES = None`.


In [15]:
# Set MAX_ENTITIES to 50_000 for a fast verification run, or None for the full dataset!
MAX_ENTITIES = None

print(f"[*] Starting streaming pair generation to: {OUTPUT_TSV_PATH}")
result = generate_labeled_pairs_tsv(
    s1_tsv_path=SOURCE1_PATH,
    ground_truth_map=gt_map,
    index=blocking_index,
    output_tsv_path=OUTPUT_TSV_PATH,
    neg_ratio=NEG_RATIO,
    singleton_neg_count=2,
    chunk_size=CHUNK_SIZE,
    max_s1_entities=MAX_ENTITIES,
    random_seed=RANDOM_SEED
)

print("\n" + "=" * 65)
print("[✓] LABELED PAIR GENERATION COMPLETE!")
print("=" * 65)
print(f"Entities Processed    : {result['entities_processed']:,}")
print(f"Positive Pairs (1)    : {result['total_positives']:,}")
print(f"Negative Pairs (0)    : {result['total_negatives']:,}")
print(f"Total Labeled Pairs   : {result['total_pairs']:,}")
print(f"Total Time Taken      : {result['duration_seconds']:.2f} seconds")
print(f"Output File           : {result['output_file']}")


[*] Starting streaming pair generation to: c:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv
    Entities: 2,471 | Positives: 8,581 | Negatives: 41,420 (1220 ent/sec)
    Entities: 4,944 | Positives: 17,160 | Negatives: 82,856 (1217 ent/sec)
    Entities: 7,390 | Positives: 25,799 | Negatives: 124,217 (1217 ent/sec)
    Entities: 9,902 | Positives: 34,408 | Negatives: 165,608 (1181 ent/sec)
    Entities: 12,351 | Positives: 43,000 | Negatives: 207,038 (1167 ent/sec)
    Entities: 14,809 | Positives: 51,582 | Negatives: 248,463 (1167 ent/sec)
    Entities: 17,285 | Positives: 60,152 | Negatives: 289,899 (1152 ent/sec)
    Entities: 19,774 | Positives: 68,724 | Negatives: 331,335 (1155 ent/sec)
    Entities: 22,294 | Positives: 77,315 | Negatives: 372,749 (1156 ent/sec)
    Entities: 24,765 | Positives: 85,898 | Negatives: 414,166 (1161 ent/sec)
    Entities: 27,255 | Positives: 94,517 | Negatives: 455,555 (1158 ent/sec)
    Entities: 

## Step 6: Sanity Check & Verification
Let's read back the top rows from `labeledPairs.tsv` and verify the label balance.


In [17]:
# 1. Read first 20 rows of generated pairs
df_sample = pd.read_csv(OUTPUT_TSV_PATH, sep="\t", nrows=20)
print("Top 20 rows of labeledPairs.tsv:")
display(df_sample)

# 2. Check class balance across sample of rows
df_balance = pd.read_csv(OUTPUT_TSV_PATH, sep="\t", nrows=100_000)
print("\nLabel Distribution in generated file (Sample):")
print(df_balance["label"].value_counts())
print("\nLabel Proportions:")
print(df_balance["label"].value_counts(normalize=True).round(3))

# 3. Check file size on disk
file_size_mb = os.path.getsize(OUTPUT_TSV_PATH) / (1024 * 1024)
print(f"\nGenerated File Size: {file_size_mb:.2f} MB")


Top 20 rows of labeledPairs.tsv:


,source1_entity_id,target_entity_id,label
0,S1-925783039,S2-517291332,1
1,S1-925783039,S3-698172821,1
2,S1-925783039,S3-997698194,1
3,S1-925783039,S2-157377754,1
4,S1-925783039,S3-701082400,0
5,S1-925783039,S3-227262213,0
6,S1-925783039,S2-811764895,0
7,S1-925783039,S2-182539574,0
8,S1-925783039,S3-7810385,0
9,S1-925783039,S2-310334217,0



Label Distribution in generated file (Sample):
label
0    82840
1    17160
Name: count, dtype: int64

Label Proportions:
label
0    0.828
1    0.172
Name: proportion, dtype: float64

Generated File Size: 1179.53 MB


## Summary & Next Steps

### What We Accomplished:
1. **Pre-Filter Layer Implemented:** Country partitioning + Rarest Token inverted index + optional Trigram fallback.
2. **Deterministic & Identical Rules:** The exact same candidate retrieval rules used here for training will be deployed at test time.
3. **High-Quality Training Data:**
   - Label 1: True Ground Truth matches.
   - Label 0: Realistic hard negatives that share similar words and same country.
4. **Memory-Safe Streaming:** Safely generated labeled pairs without memory crashes or swapping.

### Next Step:
Now that `data/pre-processing/labeledPairs.tsv` is created, we can proceed to:
- Feed these entity pairs into the **Two-Tower Model** (or gradient boosted tree feature extractor).
- Train the model to score candidate pairs with high precision!
